# Chemical Composition (BWMD): from data entry to RDF

This notebook shows how to describe the chemical composition of a material
and convert that description into a standardised, machine-readable RDF graph
following the
[BWMD Ontology](https://gitlab.cc-asp.fraunhofer.de/EMI_datamanagement/bwmd_ontology).

**You only need to edit one file:** `docs/example.input.json`.  Everything
else is automatic.

---

## What the notebook does

```
example.input.json
  │  your material name and element fractions
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
```

---

## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # BWMD/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))


Schema : schemas/chemical-composition/BWMD


---
## Step 1: Describe your material

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `material_name` | yes | A name or identifier for the material |
| `elements` | yes | List of `{symbol, value, unit}`, one per element |
| `material_id` | no | Custom IRI slug (auto-derived from `material_name` if omitted) |
| `comp_id` | no | Custom IRI slug for the composition node (auto-derived if omitted) |

Element `unit` must be `"mass%"`, `"vol%"`, or `"mol%"`.  All elements in one
composition should use the same unit.

In [3]:
simplified_path = HERE / "example.input.json"
simplified = json.loads(simplified_path.read_text())

print(json.dumps(simplified, indent=2))

{
  "label": "316L Chemical Composition",
  "base_element": {
    "symbol": "Fe"
  },
  "fractions": [
    {
      "symbol": "Cr",
      "min_value": 16.0,
      "max_value": 18.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Ni",
      "min_value": 10.0,
      "max_value": 14.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Mo",
      "min_value": 2.0,
      "max_value": 3.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Mn",
      "min_value": 0.0,
      "max_value": 2.0,
      "unit": "wt.%"
    },
    {
      "symbol": "Si",
      "min_value": 0.0,
      "max_value": 0.75,
      "unit": "wt.%"
    },
    {
      "symbol": "C",
      "min_value": 0.0,
      "max_value": 0.03,
      "unit": "wt.%"
    },
    {
      "symbol": "P",
      "min_value": 0.0,
      "max_value": 0.045,
      "unit": "wt.%"
    },
    {
      "symbol": "S",
      "min_value": 0.0,
      "max_value": 0.03,
      "unit": "wt.%"
    }
  ]
}


---
## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document, the intermediate
format that carries both the data and its ontology mapping.

Node identifiers (`material_id`, `comp_id`) are derived automatically from
`material_name` as a URL-safe slug (e.g. `"316L Stainless Steel"` →
`mat-316l-stainless-steel`).  Override them in the input only when you need
a specific IRI to match an existing knowledge graph node.

You can also run the transform from the command line:
```
python -m jsonata -e ../specs/transform.simplified.jsonata -i example.input.json
```

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/BWMD/#v1.0.0",
  "type": "bwmd:ChemicalComposition",
  "label": "316L Chemical Composition",
  "base_element": {
    "type": "bwmd:BaseElementOfComposition",
    "element_symbol": "Fe"
  },
  "fractions": [
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Cr",
      "min_value": 16.0,
      "max_value": 18.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Ni",
      "min_value": 10.0,
      "max_value": 14.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Mo",
      "min_value": 2.0,
      "max_value": 3.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "element_symbol": "Mn",
      "min_value": 0.0,
      "max_value": 2.0,
      "unit_symbol": "wt.%"
    },
    {
      "type": "bwmd:WeightFraction",
      "e

---
## Step 3: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`, which maps every field name to its ontology IRI.
rdflib ≥ 7 handles JSON-LD 1.1 natively.

In [5]:
flat = schema.to_graph(simplified)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 54 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ns1: <https://www.iwm.fraunhofer.de/ontologies/bwmd-ontology#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

[] a ns1:ChemicalComposition ;
    rdfs:label "316L Chemical Composition" ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/BWMD/#v1.0.0> ;
    ns1:hasPart [ a ns1:WeightFraction ;
            ns1:hasDoubleLiteralMaxValue 2e+00 ;
            ns1:hasDoubleLiteralMinValue 0e+00 ;
            ns1:hasUnitSymbol "wt.%" ;
            ns1:refersToElementSymbol "Mn" ],
        [ a ns1:WeightFraction ;
            ns1:hasDoubleLiteralMaxValue 4.5e-02 ;
            ns1:hasDoubleLiteralMinValue 0e+00 ;
            ns1:hasUnitSymbol "wt.%" ;
            ns1:refersToElementSymbol "P" ],
        [ a ns1:WeightFraction ;
            ns1:hasDoubleLiteralMaxValue 3e-02 ;
           

In [6]:
# Optional: save to file
out_ttl = HERE / "output_316L.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_316L.ttl


---
## Step 4: Validate against the SHACL shape

The SHACL shape (`specs/shape.ttl`) checks that the graph is structurally
correct; for example, that every element fraction has a numeric value
in \[0, 100\] and an allowed unit string.

> No `inference="rdfs"` flag is needed here; all shapes target classes
> directly without relying on subclass reasoning.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


---
## Step 5: Inspect the graph

The cell below uses SPARQL to retrieve all element fractions from the graph
and display them in a table.

You do not need to understand SPARQL to use this notebook. Just run the cell
and check that the output matches what you entered.

In [8]:
SPARQL_COMP = """
PREFIX bwmd: <https://www.iwm.fraunhofer.de/ontologies/bwmd-ontology#>

SELECT ?element ?min_value ?max_value ?unit
WHERE {
  ?comp a bwmd:ChemicalComposition ;
        bwmd:hasPart ?frac .
  ?frac a bwmd:WeightFraction ;
        bwmd:refersToElementSymbol ?element ;
        bwmd:hasDoubleLiteralMinValue ?min_value ;
        bwmd:hasDoubleLiteralMaxValue ?max_value ;
        bwmd:hasUnitSymbol ?unit .
}
ORDER BY ?element
"""

rows = list(flat.query(SPARQL_COMP))
if rows:
    unit = str(rows[0].unit)
    BWMD = rdflib.Namespace("https://www.iwm.fraunhofer.de/ontologies/bwmd-ontology#")
    base_nodes = list(flat.subjects(rdflib.RDF.type, BWMD.BaseElementOfComposition))
    if base_nodes:
        base_sym = flat.value(base_nodes[0], BWMD.refersToElementSymbol)
        print(f"Base element: {base_sym}\n")
    print(f"  {'Element':<8}  {'Min':>8}  {'Max':>8}  Unit")
    print("  " + "-" * 36)
    for r in rows:
        min_v = f"{float(r.min_value):>8.4g}"
        max_v = f"{float(r.max_value):>8.4g}"
        print(f"  {str(r.element):<8}  {min_v}  {max_v}  {unit}")
else:
    print("No element fractions found.")

Base element: Fe

  Element        Min       Max  Unit
  ------------------------------------
  C                0      0.03  wt.%
  Cr              16        18  wt.%
  Mn               0         2  wt.%
  Mo               2         3  wt.%
  Ni              10        14  wt.%
  P                0     0.045  wt.%
  S                0      0.03  wt.%
  Si               0      0.75  wt.%


---
## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the material name and element fractions |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Element fractions are extracted from the graph to confirm correctness |

To use your own material, edit `docs/example.input.json` and re-run all cells.

---

## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/3_schema-format.md): how to write your own schema
- [Simplified input guide](../../../docs/simplified-input-guide.md): end-to-end workflow